# Smoke test

Proves the remote kernel actually works before any research time is spent on it.
Run top to bottom. Every cell should print something and none should raise.

If cell 1 prints your *laptop's* hostname, you are on the local kernel — go back and
select the remote one.

In [1]:
# 1. Where am I, and is there a GPU?
import os, socket, subprocess, sys

print("hostname :", socket.gethostname())
print("python   :", sys.version.split()[0])
print("WORKSPACE:", os.environ.get("WORKSPACE", "(unset — local kernel?)"))
print("HF_HOME  :", os.environ.get("HF_HOME", "(unset)"))
print()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)

hostname : 35c26532485a
python   : 3.12.14
WORKSPACE: /workspace
HF_HOME  : /workspace/hf_cache

NVIDIA GeForce RTX 4090, 24564 MiB



In [14]:
%pip install jupyter-widgets


ERROR: Could not find a version that satisfies the requirement jupyter-widgets (from versions: none)
ERROR: No matching distribution found for jupyter-widgets
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 2. torch sees CUDA
import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "no CUDA — wrong image or a bad host; just down and retry"
print(torch.cuda.get_device_name(0))

# A real allocation, not just a capability check.
x = torch.randn(4096, 4096, device="cuda")
print("matmul ok:", (x @ x).sum().item() is not None)
del x; torch.cuda.empty_cache()

torch 2.13.0+cu126 | cuda True
NVIDIA GeForce RTX 4090
matmul ok: True


In [3]:
# 3. Project config resolves
from nandaproj import config

cfg = config.get_model_config()
config.ensure_dirs()
print(cfg)
print("device:", config.get_device())
print("cache :", config.HF_CACHE)

ModelConfig(name='google/gemma-3-270m-it', n_params='270M', dtype='bfloat16', gated=True, lens_id='gemma-3-270m-it')
device: cuda
cache : /workspace/hf_cache


In [5]:
# 4. TransformerLens: load, forward pass, cached activations
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained(cfg.name, device="cuda")
print(f"{cfg.name}: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, "
      f"d_model={model.cfg.d_model}")

prompt = "When John and Mary went to the store, John gave a drink to"
logits, cache = model.run_with_cache(prompt)
print("logits:", tuple(logits.shape))
print("top prediction:", repr(model.to_string(logits[0, -1].argmax())))

pattern = cache["pattern", 0]
print("layer-0 attention pattern:", tuple(pattern.shape))

/tmp/ipykernel_3125/3081032949.py:4: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained(cfg.name, device="cuda")


config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

Loaded pretrained model google/gemma-3-270m-it into HookedTransformer
google/gemma-3-270m-it: 18 layers, 4 heads, d_model=640
logits: (1, 15, 262144)
top prediction: ' Mary'
layer-0 attention pattern: (1, 4, 15, 15)


In [7]:
# 5. Plotting round-trips through the tunnel
from nandaproj.viz import imshow

imshow(pattern[0,0], title="L0H0 attention", xaxis="key pos", yaxis="query pos")

In [8]:
# 6. nnsight imports and can trace
import nnsight
from nnsight import LanguageModel

print("nnsight", nnsight.__version__)
lm = LanguageModel("openai-community/gpt2", device_map="cuda")
with lm.trace("The Eiffel Tower is in"):
    hidden = lm.transformer.h[6].output[0].save()
print("traced hidden state:", tuple(hidden.shape))

nnsight 0.7.0


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

traced hidden state: (7, 768)


In [11]:
acts.shape

torch.Size([1, 15, 640])

In [12]:
# 7. SAELens loads a pretrained SAE
# gpt2-small-res-jb is ungated, so this works without an HF token.
# Swap to a Gemma Scope release once HF_TOKEN is set and you move to Gemma-2-2B.
from sae_lens import SAE

sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device="cuda",
)
sae = sae[0] if isinstance(sae, tuple) else sae
print("SAE d_in:", sae.cfg.d_in, "| d_sae:", sae.cfg.d_sae)

acts = cache["blocks.7.hook_resid_pre"]
features = sae.encode(acts)
print("feature acts:", tuple(features.shape),
      "| L0 (avg live features):", (features > 0).float().sum(-1).mean().item())

SAE d_in: 768 | d_sae: 24576


/venv/main/lib/python3.12/site-packages/sae_lens/saes/sae.py:254: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


RuntimeError: The size of tensor a (640) must match the size of tensor b (768) at non-singleton dimension 2

If all seven cells ran, the environment is green: remote GPU, TransformerLens, nnsight,
SAELens, plotting, and the shared kernel.

**Now run `just down`.**